# Numerical Evidence for log-convexity of purity as a function of loss

In [ ]:
import numpy as np
from scipy.sparse import diags_array
from scipy.optimize import minimize, basinhopping

np.set_printoptions(precision=3, linewidth=200, suppress=True)

### Ladder operator definition

In [3]:
NMAX = 10

def lowering_operator(nmax=NMAX):
    return diags_array(np.sqrt(np.arange(1, nmax + 1)), offsets=1, shape=(nmax + 1, nmax +  1))

A = lowering_operator(NMAX).toarray()
ADAG = A.T

print(A)

[[0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    1.414 0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    1.732 0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    2.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    2.236 0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    2.449 0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    2.646 0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    2.828 0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    3.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    3.162]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]]


### Purity formulas and derivatives

Loss equation:

\begin{equation}
    T \partial_T \rho = \left ( a^\dagger a \rho + \rho a^\dagger a \right )/2 -a \rho a^\dagger
\end{equation}

Purity definition:

\begin{equation}
    \mathcal{P}(\rho) = \mathrm{Tr}[{\rho}^2]
\end{equation}

Purity first derivative:

\begin{equation}
   T \partial_T \mathcal{P}(\rho) = 2 (\mathrm{Tr}[a^\dagger a \rho^2] - \mathrm{Tr}[a \rho a^\dagger \rho])
\end{equation}

Purity second derivative:

\begin{equation}
    T^2 {\partial_T}^2 \mathcal{P}(\rho) = 2 (
        \mathrm{Tr}[a^{\dagger 2} a^2 \rho^2]
        + 2 \mathrm{Tr}[a^\dagger a \rho a^\dagger a \rho]
        + \mathrm{Tr}[a^2 \rho a^{\dagger 2} \rho]
        - 4 \mathrm{Re} \mathrm{Tr} [\rho a^\dagger a^2 \rho a^\dagger]
    )
\end{equation}

Objective:

\begin{equation}
    \mathcal{P}(\rho) (\partial_T)^2 \log \mathcal{P}(\rho) = \mathcal{P}(\rho) (\partial_T)^2 \mathcal{P}(\rho) - (\partial_T \mathcal{P}(\rho))^2
\end{equation}

In [ ]:
def purity(rho):
    return np.sum(np.conj(rho) * rho, axis=(-1, -2))


def purity_first_derivative(rho, a=A, adag=ADAG):
    return 2 * (
        np.trace(a @ rho @ rho @ adag, axis1=-1, axis2=-2) - 
        np.trace(a @ rho @ adag @ rho, axis1=-1, axis2=-2)
    )


def purity_second_derivative(rho, a=A, adag=ADAG):
    return 2 * (
        np.trace(a @ a @ rho @ adag @ adag @ rho, axis1=-1, axis2=-2) + 
        2 * np.trace((a @ rho @ adag @ a @ rho @ adag), axis1=-1, axis2=-2) +
        np.trace(a @ a @ rho @ rho @ adag @ adag, axis1=-1, axis2=-2) -
        4 * np.real(np.trace(rho @ adag @ a @ a @ rho @ adag, axis1=-1, axis2=-2))
    )


def objective(rho, a=A, adag=ADAG):
    return purity_second_derivative(rho, a, adag) * purity(rho) - purity_first_derivative(rho, a, adag)**2

### Random search

In [9]:
RNG = np.random.default_rng(seed=12345)

def random_rho(shape=(), nmax=NMAX, rng=RNG):
    x = 2 * rng.random((*shape, nmax + 1, nmax + 1)) - 1 + 1j * (2 * rng.random((*shape, nmax + 1, nmax + 1)) - 1)
    y = np.swapaxes(np.conj(x), -1, -2) @ x
    return y / (np.trace(y, axis1=-1, axis2=-2)[:, None, None])

random_rhos = random_rho((10000,))
np.min(np.real(objective(random_rhos)))

np.float64(0.8321458956651124)

### Minimization

In [ ]:
def objective_cholesky(xvec, a=A, adag=ADAG):
    x = xvec.view(np.complex128).reshape(a.shape)
    rho = np.conj(x).T @ x / np.sum(np.conj(x) * x)
    return np.real(objective(rho, a, adag))

minimize(objective_cholesky, np.ones(2 * ((NMAX + 1)**2)))

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 7.247314185307472e-07
        x: [ 2.981e+00  2.981e+00 ...  1.025e-03  5.202e-04]
      nit: 84
      jac: [ 4.351e-06 -3.159e-06 ... -5.960e-06 -2.444e-06]
 hess_inv: [[ 4.599e+03  4.588e+03 ... -8.769e+00 -8.478e+00]
            [ 4.588e+03  4.580e+03 ... -8.755e+00 -8.464e+00]
            ...
            [-8.769e+00 -8.755e+00 ...  9.877e-01  2.735e-02]
            [-8.478e+00 -8.464e+00 ...  2.735e-02  9.863e-01]]
     nfev: 22113
     njev: 91

In [105]:
basinhopping(objective_cholesky, np.ones(2 * ((NMAX + 1)**2)))

                    message: ['requested number of basinhopping iterations completed successfully']
                    success: True
                        fun: 7.247314185307472e-07
                          x: [ 2.981e+00  2.981e+00 ...  1.025e-03
                               5.202e-04]
                        nit: 100
      minimization_failures: 0
                       nfev: 5567616
                       njev: 22912
 lowest_optimization_result:  message: Optimization terminated successfully.
                              success: True
                               status: 0
                                  fun: 7.247314185307472e-07
                                    x: [ 2.981e+00  2.981e+00 ...
                                         1.025e-03  5.202e-04]
                                  nit: 84
                                  jac: [ 4.351e-06 -3.159e-06 ...
                                        -5.960e-06 -2.444e-06]
                             hess_inv: [[ 4.599